In [1]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
Heart Disease Prediction - Model Training Script
=================================================
Trains K-Nearest Neighbors (KNN) and Support Vector Machine (SVM) on the
same preprocessed dataset, following the methodology described in the
project documentation (Section 3.2 - 3.4):

- Dataset: heart.csv (Kaggle Heart Disease Dataset, 1025 rows, 14 cols)
- Preprocessing: duplicate removal, X/y split, 80:20 stratified train-test
  split (random_state=42), StandardScaler fit on training data only
- KNN: baseline k=5, then k=1..20 tested to find the best k
- SVM: RBF kernel, C=10, gamma=0.01
- Evaluation: accuracy, precision, recall, F1-score, confusion matrix
- Artifacts saved with joblib: knn_model.joblib, svm_model.joblib,
  scaler.joblib (used later by the Streamlit app)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from joblib import dump

# -------------------------------------------------------------------
# Step 1: Load dataset
# -------------------------------------------------------------------
DATA_PATH = r"heart.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    raise SystemExit(f"ERROR: Could not find dataset at {DATA_PATH}. "
                      f"Please check the file path.")

# -------------------------------------------------------------------
# Step 2: Data preprocessing (remove exact duplicate rows)
# -------------------------------------------------------------------
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows. Remaining: {after}")

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# -------------------------------------------------------------------
# Step 3: Exploratory Data Analysis (EDA)
# -------------------------------------------------------------------
print("\nTarget class distribution:")
print(df['target'].value_counts())

plt.figure(figsize=(5, 4))
sns.countplot(x='target', data=df)
plt.title('Target Class Distribution (0 = No Disease, 1 = Disease)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('eda_target_distribution.png')
plt.close()

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png')
plt.close()

print("EDA plots saved: eda_target_distribution.png, eda_correlation_heatmap.png")

# -------------------------------------------------------------------
# Step 4: Feature / target split
# -------------------------------------------------------------------
X = df.drop('target', axis=1)
y = df['target']

# -------------------------------------------------------------------
# Step 5: Train-test split (80:20, stratified, random_state=42)
# -------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# -------------------------------------------------------------------
# Step 6: Feature scaling (fit on training data only)
# -------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# MODULE A: K-Nearest Neighbors (KNN)
# =====================================================================
print("\n" + "=" * 60)
print("K-NEAREST NEIGHBORS (KNN)")
print("=" * 60)

# --- Baseline model: k = 5 ---
knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train_scaled, y_train)
y_pred_knn_baseline = knn_baseline.predict(X_test_scaled)
print(f"Baseline KNN (k=5) accuracy: "
      f"{accuracy_score(y_test, y_pred_knn_baseline):.4f}")

# --- Test k = 1 to 20 to find the best k ---
k_range = range(1, 21)
k_accuracies = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_temp.predict(X_test_scaled))
    k_accuracies.append(acc)

best_k = k_range[np.argmax(k_accuracies)]
print(f"\nBest k value: {best_k} (accuracy = {max(k_accuracies):.4f})")

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), k_accuracies, marker='o')
plt.xlabel('K value')
plt.ylabel('Accuracy')
plt.title('KNN Accuracy for k = 1 to 20')
plt.xticks(list(k_range))
plt.grid(True)
plt.tight_layout()
plt.savefig('knn_k_selection.png')
plt.close()
print("Plot saved: knn_k_selection.png")

# --- Train final KNN model using the best k ---
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn)
knn_recall = recall_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn)
knn_cm = confusion_matrix(y_test, y_pred_knn)

print(f"\nFinal KNN Model (k={best_k}) Performance:")
print(f"  Accuracy:  {knn_accuracy:.4f}")
print(f"  Precision: {knn_precision:.4f}")
print(f"  Recall:    {knn_recall:.4f}")
print(f"  F1-score:  {knn_f1:.4f}")
print(f"  Confusion Matrix:\n{knn_cm}")
print(f"\n{classification_report(y_test, y_pred_knn)}")

# =====================================================================
# MODULE B: Support Vector Machine (SVM)
# =====================================================================
print("\n" + "=" * 60)
print("SUPPORT VECTOR MACHINE (SVM)")
print("=" * 60)

# --- Hyperparameter tuning via GridSearchCV ---
# Instead of a fixed C=10, gamma=0.01, we search over a grid of
# kernel/C/gamma combinations and pick the combination that gives the
# best cross-validated accuracy on the training data.
svm_param_grid = {
    'kernel': ['rbf', 'linear', 'poly'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

svm_grid_search = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=svm_param_grid,
    cv=5,                 # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Running GridSearchCV for SVM hyperparameter tuning "
      f"({len(svm_param_grid['kernel']) * len(svm_param_grid['C']) * len(svm_param_grid['gamma'])} "
      "combinations x 5-fold CV)...")
svm_grid_search.fit(X_train_scaled, y_train)

print(f"\nBest SVM parameters found: {svm_grid_search.best_params_}")
print(f"Best cross-validation accuracy: {svm_grid_search.best_score_:.4f}")

# Save the full grid search results for reference/reporting
pd.DataFrame(svm_grid_search.cv_results_).sort_values(
    'rank_test_score'
).to_csv('svm_gridsearch_results.csv', index=False)
print("Full grid search results saved: svm_gridsearch_results.csv")

# Use the best estimator found by the grid search as the final SVM model
svm_model = svm_grid_search.best_estimator_
y_pred_svm = svm_model.predict(X_test_scaled)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
svm_cm = confusion_matrix(y_test, y_pred_svm)

print(f"\nSVM Model (tuned params: {svm_grid_search.best_params_}) Performance:")
print(f"  Accuracy:  {svm_accuracy:.4f}")
print(f"  Precision: {svm_precision:.4f}")
print(f"  Recall:    {svm_recall:.4f}")
print(f"  F1-score:  {svm_f1:.4f}")
print(f"  Confusion Matrix:\n{svm_cm}")
print(f"\n{classification_report(y_test, y_pred_svm)}")

# --- Confusion matrix plots ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(knn_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[0].set_title(f'KNN Confusion Matrix (k={best_k})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(svm_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[1].set_title('SVM Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png')
plt.close()
print("\nPlot saved: confusion_matrices.png")

# =====================================================================
# MODEL COMPARISON TABLE
# =====================================================================
comparison_df = pd.DataFrame({
    'Model': ['KNN', 'SVM'],
    'Accuracy': [knn_accuracy, svm_accuracy],
    'Precision': [knn_precision, svm_precision],
    'Recall': [knn_recall, svm_recall],
    'F1-score': [knn_f1, svm_f1]
})

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comparison_df.to_string(index=False))

better_model = 'SVM' if svm_accuracy > knn_accuracy else 'KNN'
print(f"\nBased on accuracy, {better_model} performed better on this dataset.")

comparison_df.to_csv('model_comparison.csv', index=False)

# =====================================================================
# Save models and scaler for deployment (Streamlit app)
# =====================================================================
dump(knn_model, 'knn_model.joblib')
dump(svm_model, 'svm_model.joblib')
dump(scaler, 'scaler.joblib')
dump(list(X.columns), 'feature_columns.joblib')
dump(best_k, 'best_k.joblib')
dump(svm_grid_search.best_params_, 'best_svm_params.joblib')

print("\nSaved artifacts:")
print("  - knn_model.joblib")
print("  - svm_model.joblib  (best estimator from GridSearchCV)")
print("  - scaler.joblib")
print("  - feature_columns.joblib")
print("  - best_k.joblib")
print("  - best_svm_params.joblib")
print("  - svm_gridsearch_results.csv  (full tuning results)")
print("\nTraining complete. You can now run the Streamlit app (app.py).")

Dataset loaded successfully. Shape: (1025, 14)
Removed 723 duplicate rows. Remaining: 302

Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Target class distribution:
target
1    164
0    138
Name: count, dtype: int64
EDA plots saved: eda_target_distribution.png, eda_correlation_heatmap.png

Training set size: 241
Testing set size: 61

K-NEAREST NEIGHBORS (KNN)
Baseline KNN (k=5) accuracy: 0.7869

Best k value: 20 (accuracy = 0.8689)
Plot saved: knn_k_selection.png

Final KNN Model (k=20) Performance:
  Accuracy:  0.8689
  Precision: 0.8571
  Recall:    0.9091
  F1-score:  0.8824
  Confusion Matrix:
[[23  5]
 [ 3 30]]

              precision    recall  f1-score   support

           0       0.88      0.82      0.85        28
           1       0.86      0.91      0.88        33

    accuracy         